In the previous notebook 1modeling_experiments.ipynb, we found that the Prophet model performed well in terms of forecasting rat sightings by day in Manhattan. In this notebook, we will compare Prophet against NeuralProphet to see if better performance can be obtained.

# Import Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging


Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


# Prophet

## Load and Prepare Data

In [2]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# Again, we consider 2020-01-01 to 2025-03-01.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
## Restrict to MANHATTAN
rs = rs[rs['borough']=='MANHATTAN']
## Drop the column with borough
rs = rs.drop(columns=['borough'])
## rename columns for prophet
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [3]:
rs["ds"] = pd.to_datetime(rs["ds"])
full_range = pd.date_range(start="2020-01-01", end="2025-02-28", freq="D")
# reindex and fill missing with 0
rs = rs.set_index("ds").reindex(full_range, fill_value=0).rename_axis("ds").reset_index()
rs

,ds,y
0,2020-01-01,4
1,2020-01-02,7
2,2020-01-03,16
3,2020-01-04,10
4,2020-01-05,5
...,...,...
1881,2025-02-24,19
1882,2025-02-25,20
1883,2025-02-26,15
1884,2025-02-27,16


In [4]:
date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [5]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAPE
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Convert results to a datafrane
prophet_results_df = pd.DataFrame(results)

14:37:45 - cmdstanpy - INFO - Chain [1] start processing
14:37:46 - cmdstanpy - INFO - Chain [1] done processing
14:37:46 - cmdstanpy - INFO - Chain [1] start processing
14:37:46 - cmdstanpy - INFO - Chain [1] done processing
14:37:47 - cmdstanpy - INFO - Chain [1] start processing
14:37:47 - cmdstanpy - INFO - Chain [1] done processing
14:37:47 - cmdstanpy - INFO - Chain [1] start processing
14:37:47 - cmdstanpy - INFO - Chain [1] done processing
14:37:48 - cmdstanpy - INFO - Chain [1] start processing
14:37:48 - cmdstanpy - INFO - Chain [1] done processing
14:37:49 - cmdstanpy - INFO - Chain [1] start processing
14:37:49 - cmdstanpy - INFO - Chain [1] done processing
14:37:49 - cmdstanpy - INFO - Chain [1] start processing
14:37:49 - cmdstanpy - INFO - Chain [1] done processing
14:37:50 - cmdstanpy - INFO - Chain [1] start processing
14:37:50 - cmdstanpy - INFO - Chain [1] done processing
14:37:51 - cmdstanpy - INFO - Chain [1] start processing
14:37:51 - cmdstanpy - INFO - Chain [1]

## Prophet's Results

In [6]:
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]
prophet_results_df

,fold,rmse,mape
0,0,2.636798,0.167563
1,1,3.877871,0.221328
2,2,4.432229,0.396771
3,3,5.226080,0.293793
4,4,4.959587,0.406843
5,5,5.547009,0.267177
6,6,5.520991,0.214916
7,7,3.865374,0.195284
8,8,3.760019,0.271349
9,9,4.539107,0.175248


# Neural Prophet

## Load and Prepare Data

In [7]:
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

In [8]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data 
rs = rs[rs['created_date']<'2025-02-28']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to MANHATTAN

rs = rs[rs['borough']=='MANHATTAN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

rs

,ds,y
2,2020-01-01,4
7,2020-01-02,7
12,2020-01-03,16
17,2020-01-04,10
21,2020-01-05,5
...,...,...
8900,2025-02-23,8
8905,2025-02-24,19
8910,2025-02-25,20
8915,2025-02-26,15


In [9]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28" 

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [10]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)


regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

## Optuna Hyperparameter Tuning

In [11]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 60),
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 60),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 7),
    }
    n_lags = trial.suggest_int('n_lags', 1, 60)
    epochs = trial.suggest_int('epochs', 10, 250)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 1, log=True)
    batch_size = trial.suggest_int('batch_size', 12, 248)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 3)
    fold_rmses = []
    for i, (train_idx, test_idx) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_idx].copy()
        test = rs.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        # Skip fold if too few rows
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            n_lags=n_lags,
            epochs=epochs,
            ar_reg = ar_reg,
            accelerator="auto",   # uses GPU if available
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="D", progress="off")
        future = pd.concat([
            train[['ds','y'] + existing_regressors],
            test[['ds','y']].merge(wd[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)
        
    return np.mean(fold_rmses)

study = optuna.create_study(
    direction="minimize",
    study_name="model_neural",
    storage="sqlite:///model_neural.db",
    load_if_exists=True
)
study.optimize(objective, n_trials=1)  # adjust n_trials as needed



best_params = study.best_params

print("Best Parameters", best_params)
print("Best RMSE:", study.best_value)

[I 2026-03-14 14:38:06,516] Using an existing study with name 'model_neural' instead of creating a new one.


Training: 0it [00:00, ?it/s]

Predicting: 74it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 75it [00:00, ?it/s]

...

Predicting: 91it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 91it [00:00, ?it/s]

[I 2026-03-14 15:04:22,255] Trial 75 finished with value: 4.749950484508321 and parameters: {'lag_temp_max': 53, 'lag_temp_min': 33, 'lag_snowfall': 2, 'n_lags': 18, 'epochs': 96, 'learning_rate': 0.0031627483740661977, 'batch_size': 20, 'ar_reg': 2.2427001005029674}. Best is trial 8 with value: 3.7346918055418405.


Best Parameters {'lag_temp_max': 26, 'lag_temp_min': 47, 'lag_snowfall': 1, 'n_lags': 13, 'epochs': 232, 'learning_rate': 0.04742593831115241, 'batch_size': 149, 'ar_reg': 2.101404589312347}
Best RMSE: 3.7346918055418405


In [12]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to MANHATTAN

rs = rs[rs['borough']=='MANHATTAN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

wd = pd.DataFrame(data["daily"])
wd["date"] = pd.to_datetime(wd["time"])
wd = wd.set_index("date")

## Evaluate the Model

In [13]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min','snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']


results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    train = train.dropna(subset=["y"])

    test = rs.iloc[test_index].copy()


    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          n_lags= best_params['n_lags'],
                          ar_reg=best_params['ar_reg'],
                          accelerator="auto",   # uses GPU if available
                          batch_size= best_params['batch_size']
                          )
    model = model.add_country_holidays(country_name="US")
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]

Training: 0it [00:00, ?it/s]

Predicting: 10it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 10it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Predicting: 13it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 13it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 13it [00:00, ?it/s]

## Neural Prophet's Results

In [14]:
neural_prophet_results_df

,fold,rmse,mape
0,0,2.837809,0.189877
1,1,4.389223,0.252642
2,2,4.222367,0.357147
3,3,5.742150,0.314968
4,4,5.111527,0.386820
5,5,5.696702,0.291639
6,6,6.017226,0.256127
7,7,4.211135,0.222551
8,8,3.919665,0.270639
9,9,4.670872,0.183099
